In [0]:

# Load Parquet & Filter

from pyspark.sql.functions import col, when, year as spark_year, count

# Load from Parquet
df = spark.read.parquet("/Volumes/workspace/default/museum/parquet/")

# Filter to rows where kingdom is not null
df_filtered = df.filter(col("kingdom").isNotNull())

print(f"Rows before filtering: {df.count():,}")
print(f"Rows after filtering: {df_filtered.count():,}")
print(f"Rows removed: {df.count() - df_filtered.count():,}")

# Show kingdom distribution
display(df_filtered.groupBy("kingdom").count().orderBy("count", ascending=False))

Rows before filtering: 6,152,101
Rows after filtering: 1,143,438
Rows removed: 5,008,663


kingdom,count
Animalia,1039075
Protista,64755
Plantae,39370
Chromista,153
Fungi,38
Algae,13
Anaimalia,10
Anamalia,8
Animala,5
Mollusca,4


In [0]:
#Select & Clean Columns 

from pyspark.sql.functions import col, when

# Select the most useful columns for classification
# Dropping columns that are >90% null or not relevant to kingdom prediction
selected_cols = [
    "kingdom",           # target variable
    "basisOfRecord",     # type of record
    "collectionCode",    # which collection
    "continent",         # geography
    "country",           # geography
    "year",              # when collected
    "month",             # when collected
    "lifeStage",         # biological stage
    "sex",               # biological sex
    "subDepartment",     # museum department
    "order",             # taxonomy
    "family",            # taxonomy
    "phylum",            # taxonomy
    "individualCount",   # count
    "typeStatus",        # specimen type
]

df_selected = df_filtered.select(selected_cols)

from pyspark.sql.functions import regexp_replace

# Fix typos in kingdom column
df_selected = df_selected.withColumn("kingdom",
    when(col("kingdom").isin(["Anaimalia", "Anamalia", "Animala"]), "Animalia")
    .when(col("kingdom").isin(["Mollusca", "Echinodermata", "Bacteria", 
                                "Protozoa", "Chromalveolata"]), "Animalia")
    .otherwise(col("kingdom"))
)

# Fill nulls
df_clean = df_selected \
    .fillna("Unknown", subset=["continent", "country", "lifeStage", 
                                "sex", "subDepartment", "order", 
                                "family", "phylum", "typeStatus", "month"]) \
    .fillna(0, subset=["individualCount"]) \
    .fillna("Unknown", subset=["year"])

print(f"Clean dataset rows: {df_clean.count():,}")
print(f"Columns: {len(df_clean.columns)}")
display(df_clean.limit(10))

Clean dataset rows: 1,143,438
Columns: 15


kingdom,basisOfRecord,collectionCode,continent,country,year,month,lifeStage,sex,subDepartment,order,family,phylum,individualCount,typeStatus
Animalia,PreservedSpecimen,BMNH(E),Africa,Zambia,2012,11,Unknown,Unknown,Small orders,Blattodea,Unknown,Arthropoda,1,Unknown
Animalia,PreservedSpecimen,BMNH(E),Africa,Nigeria,Unknown,Unknown,Unknown,Male,Small orders,Odonata,Chlorocyphidae,Arthropoda,1,LECTOTYPE
Animalia,PreservedSpecimen,BMNH(E),Africa,Zambia,2012,11,Unknown,Unknown,Small orders,Blattodea,Unknown,Arthropoda,1,Unknown
Animalia,PreservedSpecimen,BMNH(E),Africa,Zambia,2012,11,Unknown,Unknown,Small orders,Blattodea,Unknown,Arthropoda,1,Unknown
Animalia,PreservedSpecimen,BMNH(E),Africa,Tanzania,Unknown,Unknown,Unknown,Unknown,Small orders,Blattodea,Unknown,Arthropoda,1,Unknown
Animalia,PreservedSpecimen,BMNH(E),Africa,Tanzania,Unknown,Unknown,Unknown,Unknown,Small orders,Blattodea,Unknown,Arthropoda,1,Unknown
Animalia,PreservedSpecimen,BMNH(E),Africa,Zambia,2012,11,Unknown,Unknown,Small orders,Blattodea,Unknown,Arthropoda,1,Unknown
Animalia,PreservedSpecimen,BMNH(E),Africa,Tanzania,Unknown,Unknown,Unknown,Unknown,Small orders,Blattodea,Unknown,Arthropoda,1,Unknown
Animalia,PreservedSpecimen,BMNH(E),Africa,Tanzania,Unknown,Unknown,Unknown,Unknown,Small orders,Blattodea,Unknown,Arthropoda,1,Unknown
Animalia,PreservedSpecimen,BMNH(E),Africa,Zambia,2012,11,Unknown,Unknown,Small orders,Blattodea,Unknown,Arthropoda,1,Unknown


In [0]:
#Features for MLlib

from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

# Columns that need encoding (categorical -> numeric)
categorical_cols = ["basisOfRecord", "collectionCode", "continent", "country",
"lifeStage", "sex", "subDepartment", "order", "family", "phylum", "typeStatus", "month", "year"]

# Create StringIndexer for each categorical column
indexers = [
    StringIndexer(inputCol=c, outputCol=c + "_idx", handleInvalid="keep")
    for c in categorical_cols
]

# StringIndexer for target column (kingdom)
kingdom_indexer = StringIndexer(inputCol="kingdom", outputCol="label", handleInvalid="keep")

# Assemble all feature columns into a single vector
feature_cols = [c + "_idx" for c in categorical_cols] + ["individualCount"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="keep")

# Build and run pipeline
pipeline = Pipeline(stages=indexers + [kingdom_indexer, assembler])
pipeline_model = pipeline.fit(df_clean)
df_encoded = pipeline_model.transform(df_clean)


display(df_encoded.select("kingdom", "label", "features").limit(5))

Encoding complete
Total features: 14


kingdom,label,features
Animalia,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""2.0"",""2.0"",""43.0"",""0.0"",""0.0"",""42.0"",""436.0"",""0.0"",""2.0"",""0.0"",""11.0"",""21.0"",""1.0""]}"
Animalia,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""2.0"",""2.0"",""20.0"",""0.0"",""3.0"",""42.0"",""978.0"",""5140.0"",""2.0"",""21.0"",""0.0"",""0.0"",""1.0""]}"
Animalia,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""2.0"",""2.0"",""43.0"",""0.0"",""0.0"",""42.0"",""436.0"",""0.0"",""2.0"",""0.0"",""11.0"",""21.0"",""1.0""]}"
Animalia,0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""2.0"",""2.0"",""43.0"",""0.0"",""0.0"",""42.0"",""436.0"",""0.0"",""2.0"",""0.0"",""11.0"",""21.0"",""1.0""]}"
Animalia,0.0,"{""type"":""0"",""size"":""14"",""indices"":[""1"",""2"",""3"",""6"",""7"",""9"",""13""],""values"":[""2.0"",""2.0"",""7.0"",""42.0"",""436.0"",""2.0"",""1.0""]}"


In [0]:
#Save Prepared Data 

# Save the prepared data for model training
output_path = "/Volumes/workspace/default/museum/prepared/"

df_encoded.select("label", "features", "kingdom") \
    .write.mode("overwrite") \
    .parquet(output_path)



# Verify
df_verify = spark.read.parquet(output_path)
display(df_verify.limit(5))

Prepared data saved to: /Volumes/workspace/default/museum/prepared/
Verified row count: 1,143,438


label,features,kingdom
0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""0.0"",""2.0"",""10.0"",""1.0"",""1.0"",""0.0"",""0.0"",""0.0"",""0.0"",""6.0"",""9.0"",""33.0"",""1.0""]}",Animalia
0.0,"{""type"":""1"",""size"":null,""indices"":null,""values"":[""0.0"",""0.0"",""2.0"",""11.0"",""0.0"",""0.0"",""5.0"",""22.0"",""24.0"",""4.0"",""0.0"",""4.0"",""71.0"",""0.0""]}",Animalia
0.0,"{""type"":""0"",""size"":""14"",""indices"":[""2"",""3"",""6"",""7"",""8"",""9"",""10""],""values"":[""2.0"",""63.0"",""2.0"",""61.0"",""66.0"",""1.0"",""2.0""]}",Animalia
0.0,"{""type"":""0"",""size"":""14"",""indices"":[""6"",""7"",""8"",""9"",""12"",""13""],""values"":[""5.0"",""9.0"",""36.0"",""4.0"",""38.0"",""1.0""]}",Animalia
0.0,"{""type"":""0"",""size"":""14"",""indices"":[""6"",""7"",""8"",""9"",""10"",""12""],""values"":[""11.0"",""74.0"",""227.0"",""9.0"",""7.0"",""181.0""]}",Animalia
